In [2]:
#파일 로드
import pandas as pd

df=pd.read_csv("/Users/mugyeom/workspace/skala-intro/광주_1반_김무겸_데이터 분석을 위한 파이썬 이해 day2/sales_100k.csv")

In [3]:
#파일 탐색
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 11 columns):
 #   Column           Non-Null Count    Dtype  
---  ------           --------------    -----  
 0   order_id         1000000 non-null  int64  
 1   order_date       1000000 non-null  object 
 2   region           990000 non-null   object 
 3   category         992000 non-null   object 
 4   product_name     1000000 non-null  object 
 5   quantity         1000000 non-null  int64  
 6   unit_price       1000000 non-null  int64  
 7   payment_method   1000000 non-null  object 
 8   customer_age     1000000 non-null  int64  
 9   customer_gender  1000000 non-null  object 
 10  amount           995000 non-null   float64
dtypes: float64(1), int64(4), object(6)
memory usage: 83.9+ MB


In [4]:
df.describe()

,order_id,quantity,unit_price,customer_age,amount
count,1000000.000000,1000000.000000,1000000.000000,1000000.000000,9.950000e+05
mean,500000.500000,10.005831,250316.618029,43.490057,3.034762e+06
std,288675.278932,5.476360,144150.843629,15.010487,5.589690e+06
min,1.000000,1.000000,1001.000000,18.000000,1.004000e+03
25%,250000.750000,5.000000,125412.000000,30.000000,7.572055e+05
50%,500000.500000,10.000000,250163.000000,43.000000,1.948270e+06
75%,750000.250000,15.000000,375250.000000,57.000000,3.928346e+06
max,1000000.000000,19.000000,499999.000000,69.000000,1.373796e+08


In [5]:
df.shape

(1000000, 11)

In [6]:
df.head

<bound method NDFrame.head of         order_id  order_date region category product_name  quantity  \
0              1  2023-04-13     인천       전자      상품_8036         4   
1              2  2024-03-11     부산       전자      상품_5341        15   
2              3  2023-09-28    NaN       의류      상품_6292        16   
3              4  2023-04-17     서울       식품      상품_1670        11   
4              5  2023-03-13     서울       가구      상품_5231        16   
...          ...         ...    ...      ...          ...       ...   
999995    999996  2024-07-16     대구       식품      상품_6809         9   
999996    999997  2024-12-14     서울       도서      상품_6911         6   
999997    999998  2023-12-31     경기      스포츠      상품_2741        19   
999998    999999  2024-07-08     대전       도서      상품_1614        17   
999999   1000000  2023-03-27     대구       전자      상품_7623         3   

        unit_price payment_method  customer_age customer_gender     amount  
0           266044            포인트       

In [7]:
#IQR계산 및 이상치 제거
Q1 = df['amount'].quantile(0.25)
Q3 = df['amount'].quantile(0.75)
IQR = Q3-Q1
lo, hi = Q1-1.5*IQR, Q3+1.5*IQR

df_clean = df[df["amount"].between(lo,hi)]
print(f"제거 전: {len(df)}, 제거 후: {len(df_clean)}\n제거 행 : {len(df)-len(df_clean)}")

제거 전: 1000000, 제거 후: 973806
제거 행 : 26194


In [ ]:
#지역별 통계
pd_result_region = df_clean.groupby("region").agg(
    revenue=('amount','sum'),
    mean = ('amount', 'mean'),
    cnt=('amount','count')
    ).sort_values("revenue", ascending = False)
print(f'groupby 후 데이터 행:{region_total["cnt"].sum()+df_clean["region"].isna().sum()}\nclean 후 데이터 행:{len(df_clean)}')
print(pd_region_total)

groupby 후 데이터 행:973806
clean 후 데이터 행:973806
             revenue          mean     cnt
region                                    
서울      5.970126e+11  2.476501e+06  241071
경기      4.737745e+11  2.461793e+06  192451
부산      2.873737e+11  2.471076e+06  116295
인천      2.384407e+11  2.469941e+06   96537
대구      2.377930e+11  2.473918e+06   96120
광주      1.903844e+11  2.474678e+06   76933
대전      1.898833e+11  2.470541e+06   76859
울산      1.670818e+11  2.465606e+06   67765


In [ ]:
#카테고리별 통계
pd_result_category = df_clean.groupby("category").agg(
    revenue=('amount','sum'),
    mean = ('amount', 'mean'),
    cnt=('amount','count')
    ).sort_values("revenue", ascending = False)
print(f'groupby 후 데이터 행:{region_total["cnt"].sum()+df_clean["category"].isna().sum()}\nclean 후 데이터 행:{len(df_clean)}')
print(pd_region_total)

groupby 후 데이터 행:973806
clean 후 데이터 행:973806
               revenue          mean     cnt
category                                    
식품        2.994052e+11  2.475344e+06  120955
의류        2.991441e+11  2.476789e+06  120779
도서        2.990370e+11  2.476067e+06  120771
가구        2.987681e+11  2.466060e+06  121152
뷰티        2.985665e+11  2.472498e+06  120755
완구        2.982176e+11  2.472495e+06  120614
스포츠       2.973622e+11  2.461424e+06  120809
전자        2.966242e+11  2.467345e+06  120220


In [10]:
%pip install polars

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
#3)Polars Lazy API로 동일 집계 작성
import polars as pl

pl_result_region = (
    pl.scan_csv('/Users/mugyeom/workspace/skala-intro/광주_1반_김무겸_데이터 분석을 위한 파이썬 이해 day2/sales_100k.csv',schema_overrides={'amount':pl.Float64})
    .filter((pl.col('amount')>lo) & (pl.col('amount')<hi)).group_by('region').agg([pl.col('amount').sum().alias("revenue"),pl.col('amount').mean().alias("mean"),pl.col('amount').count().alias("cnt")])
    .sort('revenue',descending=True).collect()
)

pl_result_category = (
    pl.scan_csv('/Users/mugyeom/workspace/skala-intro/광주_1반_김무겸_데이터 분석을 위한 파이썬 이해 day2/sales_100k.csv',schema_overrides={'amount':pl.Float64})
    .filter((pl.col('amount')>lo) & (pl.col('amount')<hi)).group_by('category').agg([pl.col('amount').sum().alias("revenue"),pl.col('amount').mean().alias("mean"),pl.col('amount').count().alias("cnt")])
    .sort('revenue',descending=True).collect()
)

print(pl_result_region)
print(pl_result_category)


shape: (9, 4)
┌────────┬───────────┬──────────┬────────┐
│ region ┆ revenue   ┆ mean     ┆ cnt    │
│ ---    ┆ ---       ┆ ---      ┆ ---    │
│ str    ┆ f64       ┆ f64      ┆ u32    │
╞════════╪═══════════╪══════════╪════════╡
│ 서울   ┆ 5.9701e11 ┆ 2.4765e6 ┆ 241071 │
│ 경기   ┆ 4.7377e11 ┆ 2.4618e6 ┆ 192451 │
│ 부산   ┆ 2.8737e11 ┆ 2.4711e6 ┆ 116295 │
│ 인천   ┆ 2.3844e11 ┆ 2.4699e6 ┆ 96537  │
│ 대구   ┆ 2.3779e11 ┆ 2.4739e6 ┆ 96120  │
│ 광주   ┆ 1.9038e11 ┆ 2.4747e6 ┆ 76933  │
│ 대전   ┆ 1.8988e11 ┆ 2.4705e6 ┆ 76859  │
│ 울산   ┆ 1.6708e11 ┆ 2.4656e6 ┆ 67765  │
│ null   ┆ 2.4113e10 ┆ 2.4668e6 ┆ 9775   │
└────────┴───────────┴──────────┴────────┘
shape: (9, 4)
┌──────────┬───────────┬──────────┬────────┐
│ category ┆ revenue   ┆ mean     ┆ cnt    │
│ ---      ┆ ---       ┆ ---      ┆ ---    │
│ str      ┆ f64       ┆ f64      ┆ u32    │
╞══════════╪═══════════╪══════════╪════════╡
│ 식품     ┆ 2.9941e11 ┆ 2.4753e6 ┆ 120955 │
│ 의류     ┆ 2.9914e11 ┆ 2.4768e6 ┆ 120779 │
│ 도서     ┆ 2.9904e11 ┆ 2.4761e6 

In [ ]:
#4)DuckDB SQL

import duckdb

du_result_region = duckdb.sql(f"""
                    SELECT region,
                    SUM(amount) AS revenue,
                    AVG(amount) AS mean,
                    COUNT(*) AS cnt
                    FROM '/Users/mugyeom/workspace/skala-intro/광주_1반_김무겸_데이터 분석을 위한 파이썬 이해 day2/sales_100k.csv'
                    WHERE amount BETWEEN {lo} AND {hi}
                    GROUP BY region
                    ORDER BY revenue DESC"""
).df()

du_result_category = duckdb.sql(f"""
                    SELECT category,
                    SUM(amount) AS revenue,
                    AVG(amount) AS mean,
                    COUNT(*) AS cnt
                    FROM '/Users/mugyeom/workspace/skala-intro/광주_1반_김무겸_데이터 분석을 위한 파이썬 이해 day2/sales_100k.csv'
                    WHERE amount BETWEEN {lo} AND {hi}
                    GROUP BY category
                    ORDER BY revenue DESC"""
).df()

print(du_result_region)
print(du_result_category)

  region       revenue          mean     cnt
0     서울  5.970126e+11  2.476501e+06  241071
1     경기  4.737745e+11  2.461793e+06  192451
2     부산  2.873737e+11  2.471076e+06  116295
3     인천  2.384407e+11  2.469941e+06   96537
4     대구  2.377930e+11  2.473918e+06   96120
5     광주  1.903844e+11  2.474678e+06   76933
6     대전  1.898833e+11  2.470541e+06   76859
7     울산  1.670818e+11  2.465606e+06   67765
8   None  2.411277e+10  2.466779e+06    9775
  category       revenue          mean     cnt
0       식품  2.994052e+11  2.475344e+06  120955
1       의류  2.991441e+11  2.476789e+06  120779
2       도서  2.990370e+11  2.476067e+06  120771
3       가구  2.987681e+11  2.466060e+06  121152
4       뷰티  2.985665e+11  2.472498e+06  120755
5       완구  2.982176e+11  2.472495e+06  120614
6      스포츠  2.973622e+11  2.461424e+06  120809
7       전자  2.966242e+11  2.467345e+06  120220
8     None  1.873184e+10  2.416699e+06    7751


In [ ]:
#세 도구 성능 비교1 pandas는 전처리가 끝난 데이터를 바로써 빠르지만 polars와 duckdb는 파싱과정이 포함되 길게 나옴
t_pandas = %timeit -o df_clean.groupby("region").agg(revenue=('amount','sum'), mean=('amount','mean'), cnt=('amount','count'))

t_polars = %timeit -o (pl.scan_csv("/Users/mugyeom/workspace/skala-intro/광주_1반_김무겸_데이터 분석을 위한 파이썬 이해 day2/sales_100k.csv", schema_overrides={'amount': pl.Float64}).filter(pl.col('amount').is_between(lo, hi)).group_by('region').agg([pl.col('amount').sum().alias("revenue"), pl.col('amount').mean().alias("mean"), pl.col('amount').count().alias("cnt")]).sort('revenue', descending=True).collect())

t_duckdb = %timeit -o duckdb.sql(f"""SELECT region, SUM(amount) AS revenue, AVG(amount) AS mean, COUNT(*) AS cnt FROM '{"/Users/mugyeom/workspace/skala-intro/광주_1반_김무겸_데이터 분석을 위한 파이썬 이해 day2/sales_100k.csv"}' WHERE amount BETWEEN {lo} AND {hi} AND region IS NOT NULL GROUP BY region ORDER BY revenue DESC""").df()

compare = pd.DataFrame({
    "engine": ["pandas", "polars", "duckdb"],
    "avg_sec": [t_pandas.average, t_polars.average, t_duckdb.average]
})
print(compare)

23.4 ms ± 250 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)
30.3 ms ± 305 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)
117 ms ± 707 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)
   engine   avg_sec
0  pandas  0.023361
1  polars  0.030287
2  duckdb  0.116611


In [ ]:
#세 도구 성능 비교2 공정한 비교를 위해 pandas에도 파싱과정을 포함
t_pandas_full = %timeit -o (pd.read_csv("/Users/mugyeom/workspace/skala-intro/광주_1반_김무겸_데이터 분석을 위한 파이썬 이해 day2/sales_100k.csv").pipe(lambda df: df[df['amount'].between(lo, hi)]).groupby("region").agg(revenue=('amount','sum'), mean=('amount','mean'), cnt=('amount','count')))

t_polars = %timeit -o (pl.scan_csv("/Users/mugyeom/workspace/skala-intro/광주_1반_김무겸_데이터 분석을 위한 파이썬 이해 day2/sales_100k.csv", schema_overrides={'amount': pl.Float64}).filter(pl.col('amount').is_between(lo, hi)).group_by('region').agg([pl.col('amount').sum().alias("revenue"), pl.col('amount').mean().alias("mean"), pl.col('amount').count().alias("cnt")]).sort('revenue', descending=True).collect())

t_duckdb = %timeit -o duckdb.sql(f"""SELECT region, SUM(amount) AS revenue, AVG(amount) AS mean, COUNT(*) AS cnt FROM '{"/Users/mugyeom/workspace/skala-intro/광주_1반_김무겸_데이터 분석을 위한 파이썬 이해 day2/sales_100k.csv"}' WHERE amount BETWEEN {lo} AND {hi} AND region IS NOT NULL GROUP BY region ORDER BY revenue DESC""").df()

compare = pd.DataFrame({
    "engine": ["pandas", "polars", "duckdb"],
    "avg_sec": [t_pandas_full.average, t_polars.average, t_duckdb.average]
})
print(compare)

426 ms ± 960 µs per loop (mean ± std. dev. of 7 runs, 1 loop each)
30.1 ms ± 414 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)
119 ms ± 3.26 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
   engine   avg_sec
0  pandas  0.426401
1  polars  0.030099
2  duckdb  0.119448
